## NEP_Phon3

### 声子寿命

In [ ]:
from ase.io import read
from hiphive import ForceConstantPotential
from phon import calculate_third_order_properties

def main():
    """
    主函数
    """
    unit_cell = read("./POSCAR")
    primitive_matrix = 'auto'
    supercell_matrix = [4, 4, 4]
    supercell_matrix_nep = [4, 4, 4]
    temp = 400
    fcp = ForceConstantPotential.read('fcps/ehm_T{}.fcp'.format(temp))
    
    calculate_third_order_properties(unit_cell, primitive_matrix, supercell_matrix, fcp, temp, "phon_lifetimes.png")
            
if __name__ == '__main__':
    main()


phono3py --dim="4 4 4" --fc3 --band "0.5 0.5 0.5 0 0 0 0 0 0 0.5 0 0.5" --band_points 101 --fc2 --br --mesh="20 20 20" --ts="400"

### 频率自能谱

In [1]:
import h5py

with h5py.File('kappa-m202020.hdf5', 'r') as f:
    frequency = f['frequency'][:]
    gamma = f['gamma'][:]
    qpoint = f['qpoint'][:]
    grid_point = f['grid_point'][:]
    print(list(f.keys()))

['boundary_mfp', 'frequency', 'gamma', 'grid_point', 'group_velocity', 'gv_by_gv', 'heat_capacity', 'kappa', 'kappa_unit_conversion', 'mesh', 'mode_kappa', 'qpoint', 'temperature', 'version', 'weight']


In [2]:
print(f"grid_point shape: {grid_point.shape}")

grid_point shape: (781,)


frequency shape: (781, 6)

gamma shape: (1, 781, 6)

grid_point shape: (781,)

qpoint shape: (781, 3)

intensity shape: (781, 6)

kpoints_rel shape: (40, 3) 

kpoints_lincoord shape: (40,)

In [2]:
import numpy as np

def lorentzian(omega, omega_0, gamma):
    """计算洛伦兹谱函数，避免除以零"""
    return (1 / np.pi) * (gamma / ((omega - omega_0)**2 + gamma**2 + 1e-10))

def calculate_intensity(frequency, gamma, mode_count):
    """计算声子的强度，并调整积分范围和分辨率"""
    num_qpoints, num_modes = frequency.shape
    intensity = np.zeros((num_qpoints, num_modes))

    for i in range(num_qpoints):
        for j in range(num_modes):
            omega_0 = frequency[i, j]
            gamma_value = gamma[0, i, j]

            # 根据最大和最小频率，合理设定积分范围 omega_range，通常可以设定为每个模式的频率 omega_0 的上下几倍 gamma 的范围内。避免在频率接近 0 或负值时出现错误。
            # omega_range 的分辨率决定了谱函数计算的精度。根据最大 gamma 值和频率范围，动态调整分辨率。通常分辨率可以设定为 gamma 的十分之一到百分之一，以确保计算的精度。
            if gamma_value > 0:
                omega_min = max(omega_0 - 5 * gamma_value, -0.1)  # 避免负值过大
                omega_max = omega_0 + 5 * gamma_value
                omega_range = np.linspace(omega_min, omega_max, 1000)
            else:
                omega_range = np.array([omega_0])  # gamma=0时，不进行积分

            # 计算谱函数并积分
            A = lorentzian(omega_range, omega_0, gamma_value)
            intensity[i, j] = np.trapz(A, omega_range) if gamma_value > 0 else 0.0

    return intensity

intensity = calculate_intensity(frequency, gamma, mode_count=6)

# 计算最大值和最小值
max_intensity = np.max(intensity)
min_intensity = np.min(intensity)

print("Maximum Intensity:", max_intensity)
print("Minimum Intensity:", min_intensity)
print(f"intensity shape: {intensity.shape}")

Maximum Intensity: 0.8743339951112478
Minimum Intensity: 0.0
intensity shape: (781, 6)


In [3]:
from ase.io import read

def get_manual_kpoints(structure, high_symmetry_points, num=30):
    """
    基于手动定义的高对称点生成k点路径
    high_symmetry_points: 包含点坐标和标签的列表
    例如：[{'label': 'X', 'coords': [0.5, 0.0, 0.5]}, {'label': 'L', 'coords': [0.5, 0.5, 0.5]}]
    """
    kpoints_rel, kpoints_lincoord, labels = [], [], []
    current_distance = 0
    
    for i, point in enumerate(high_symmetry_points[:-1]):
        start, end = np.array(point['coords']), np.array(high_symmetry_points[i + 1]['coords'])
        kpoint_segment = np.linspace(start, end, num=num)
        kpoints_rel.extend(kpoint_segment)
        
        segment_length = np.linalg.norm(end - start)
        kpoints_lincoord.extend(np.linspace(current_distance, current_distance + segment_length, num=num))
        
        current_distance += segment_length
        labels.extend([point['label']] + [''] * (num-1))
        if i == len(high_symmetry_points) - 2:
            labels[-1] = high_symmetry_points[i + 1]['label']
        labels = ['$\Gamma$' if label == 'GAMMA' else label for label in labels]
    
    return kpoints_rel, kpoints_lincoord, labels

high_symmetry_points = [
    {'label': 'L', 'coords': [0.5, 0.5, 0.5]},
    {'label': 'GAMMA', 'coords': [0.0, 0.0, 0.0]},
    {'label': 'X', 'coords': [0.5, 0.0, 0.5]},
]

unit_cell = read("./POSCAR")
kpoints_rel, kpoints_lincoord, labels  = get_manual_kpoints(unit_cell, high_symmetry_points, num=20)
len(kpoints_rel), len(kpoints_lincoord), len(labels)

(40, 40, 40)

In [39]:
import numpy as np
from scipy.spatial import distance

def project_qpoint_to_path(qpoints, path_points):
    """将 q-point 投影到路径上"""
    projected_qpoints = []
    for q in qpoints:
        # 计算 q-point 到路径上所有点的距离
        distances = distance.cdist([q], path_points)
        # 找到距离最小的路径点
        nearest_index = np.argmin(distances)
        projected_qpoints.append(path_points[nearest_index])
    return np.array(projected_qpoints)

# 投影 q-point 到路径上
path_qpoints = project_qpoint_to_path(qpoint, kpoints_rel)

print("Projected q-points on path:\n", path_qpoints)
path_qpoints.shape

Projected q-points on path:
 [[0.         0.         0.        ]
 [0.02631579 0.         0.02631579]
 [0.05263158 0.         0.05263158]
 ...
 [0.47368421 0.47368421 0.47368421]
 [0.5        0.5        0.5       ]
 [0.         0.         0.        ]]


(781, 3)

In [40]:
def count_unique_rows(arr):
    """计算数组中唯一行的数量"""
    # 使用 numpy.unique 找到唯一的行
    unique_rows = np.unique(arr, axis=0)
    # 返回唯一行的数量
    return unique_rows.shape[0]

# 计算 path_qpoints 中唯一行的数量
num_unique_rows = count_unique_rows(path_qpoints)

print(f"Number of unique rows in path_qpoints: {num_unique_rows}")

Number of unique rows in path_qpoints: 34


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.interpolate import griddata

# 假设数据
qpoint = np.random.rand(781, 3)  # 网格上的 q-points, 形状 (781, 3)
frequency = np.random.rand(781, 6)  # 对应的频率, 形状 (781, 6)
intensity = np.random.rand(781, 6)  # 对应的强度, 形状 (781, 6)
kpoints_rel = np.random.rand(40, 3)  # 路径上的 q-points, 形状 (40, 3)
kpoints_lincoord = np.linspace(0, 1, 40)  # 路径坐标, 形状 (40,)

# 插值
frequency_interp = griddata(qpoint, frequency, kpoints_rel, method='linear')
intensity_interp = griddata(qpoint, intensity, kpoints_rel, method='linear')

# 创建网格
X, Y = np.meshgrid(kpoints_lincoord, np.arange(frequency_interp.shape[1]))

# 创建频率和强度的图像数据
Z_frequency = frequency_interp.T  # 转置以匹配网格
Z_intensity = intensity_interp.T  # 转置以匹配网格

# 绘制频率数据
plt.figure(figsize=(10, 6))
plt.pcolormesh(X, Y, Z_frequency, shading='auto', norm=LogNorm(vmin=Z_frequency.min(), vmax=Z_frequency.max()), cmap='afmhot')
plt.colorbar(label='Frequency')
plt.xlabel('K-Point Linear Coordinate')
plt.ylabel('Mode Index')
plt.title('Frequency vs K-Point Linear Coordinate')

# 绘制强度数据
plt.figure(figsize=(10, 6))
plt.pcolormesh(X, Y, Z_intensity, shading='auto', norm=LogNorm(vmin=Z_intensity.min(), vmax=Z_intensity.max()), cmap='afmhot')
plt.colorbar(label='Intensity')
plt.xlabel('K-Point Linear Coordinate')
plt.ylabel('Mode Index')
plt.title('Intensity vs K-Point Linear Coordinate')

plt.show()


In [ ]:
x = qpoint
y = frequency
gz = intensity
# add a little bit so that the logscale does not go nuts
gz=gz+1E-2
# for plotting, turn the axes into 2d arrays
gx, gy = np.meshgrid(x,y)
# x-ticks
xt = np.array(f.get('q_ticks'))
# labels for the x-ticks
xl = f.attrs.get('q_tick_labels').split()
# label for y-axis

plt.pcolormesh(gx, gy, gz, norm=LogNorm(vmin=gz.min(), vmax=gz.max()), cmap='afmhot')
# set the limits of the plot to the limits of the data
plt.axis([x.min(), x.max(), y.min(), y.max()])
plt.xticks(xt,xl)

plt.tight_layout()
plt.show()

In [ ]:
import phono3py
from phono3py import Phono3py
from phono3py.file_IO import read_fc2_from_hdf5, read_fc3_from_hdf5
from phonopy.structure.atoms import PhonopyAtoms
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from ase.io import read

# 从 POSCAR 文件读取结构
structure = read("POSCAR")  # 你的POSCAR文件路径

# 创建 PhonopyAtoms 对象
phonopy_atoms = PhonopyAtoms(
    symbols=structure.symbols(),
    cell=structure.cell,
    scaled_positions=structure.get_scaled_positions(),
    pbc=[True, True, True]
)

# 定义超胞矩阵和 k-point 网格
supercell_matrix = [4, 4, 4]  # 超胞矩阵
mesh = [20, 20, 20]  # k-point 网格
temperature = 400  # 计算温度 (单位: K)

# 初始化 Phono3py 对象
ph3 = Phono3py(
    phonopy_atoms,
    supercell_matrix=supercell_matrix,
    primitive_matrix="auto",
    phonon_supercell_matrix=[6, 6, 6]
)

# ph3.save("phono3py_disp.yaml")

# ph3.generate_displacements()

# ph3.phonon_supercell_matrix
# ph3.phonon_dataset
# ph3.phonon_forces
# ph3.phonon_supercells_with_displacements

# 加载二阶和三阶力常数
ph3.fc2 = read_fc2_from_hdf5("fc2.hdf5")
ph3.fc3 = read_fc3_from_hdf5("fc3.hdf5")

# 设置网格并初始化声子-声子相互作用
ph3.mesh_numbers = mesh
phonon.init_phph_interaction()

# 运行网格计算，获取频率
ph3.run_mesh()
mesh_data = ph3.get_mesh()  # 获取网格数据

# 提取 Gamma 点（或其他指定点）的频率
gamma_point_index = np.argwhere(np.all(mesh_data[0] == 0, axis=1))[0][0]  # 找到 Gamma 点的索引
frequencies = mesh_data[1][gamma_point_index]  # 提取频率

# 计算频谱函数
phonon.run_spectral_function(
    grid_points=[gamma_point_index],  # 使用 Gamma 点的索引
    temperatures=[temperature],
    num_frequency_points=100
)

# 提取频谱函数数据
spectral_function = phonon.spectral_functions[0]

# 创建频谱图
plt.figure(figsize=(8, 6))
q_values = np.linspace(0, 1, len(frequencies))
gx, gy = np.meshgrid(q_values, frequencies)
plt.pcolormesh(gx, gy, spectral_function, norm=LogNorm(vmin=spectral_function.min(), vmax=spectral_function.max()), cmap='afmhot')
plt.colorbar(label="Spectral function")
plt.xlabel("Wave vector")
plt.ylabel("Frequency (THz)")
plt.tight_layout()
plt.savefig("./spectral_func.png", bbox_inches='tight')
